# NOAA/NCEI HazEL API — Exploration Notebook (Fixed)

**Bugs fixed from v1:**
1. `fetch_ncei_earthquakes()` now paginates correctly — the API returns 25 records per page by default, so without pagination you only ever got the first page.
2. The merge function now uses `eqMagnitude` (the actual NCEI column name) instead of `magnitude`.
3. The `time` column is built before the merge runs.

**API base URL:** `https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/earthquakes`  
**No API key required.**

In [28]:
import requests
import pandas as pd
import numpy as np
import json

BASE_URL = "https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/earthquakes"

## 1. Check the pagination structure

The API response has these keys: `items`, `page`, `totalPages`, `itemsPerPage`, `totalItems`.
We need to loop through all pages to get the full dataset.

In [29]:
# Peek at one page to understand the pagination fields
r = requests.get(BASE_URL, params={"minYear": 2020, "maxYear": 2020})
r.raise_for_status()
data = r.json()

print("Response keys:     ", list(data.keys()))
print("Items on this page:", len(data["items"]))
print("Total items:       ", data["totalItems"])   
print("Total pages:       ", data["totalPages"])
print("Items per page:    ", data["itemsPerPage"])

Response keys:      ['items', 'page', 'totalPages', 'itemsPerPage', 'totalItems']
Items on this page: 33
Total items:        33
Total pages:        1
Items per page:     200


## 2.fetch function

In [30]:
def fetch_ncei_earthquakes(min_year: int, max_year: int, min_magnitude: float = 0) -> pd.DataFrame:
    params = {
        "minYear": min_year,
        "maxYear": max_year,
        "page": 1,
    }
    if min_magnitude > 0:
        params["minMagnitude"] = min_magnitude

    all_items = []
    page = 1

    while True:
        params["page"] = page
        r = requests.get(BASE_URL, params=params)
        r.raise_for_status()
        data = r.json()

        items = data.get("items", [])
        all_items.extend(items)

        total_pages = data.get("totalPages", 1)
        print(f"  Page {page}/{total_pages} — {len(all_items)} records so far", end="\r")

        if page >= total_pages:
            break
        page += 1

    print(f"\nDone. Fetched {len(all_items)} total records ({min_year}–{max_year})")
    return pd.DataFrame(all_items)

## 3. Fetch the full 2000–2024 dataset

In [31]:
df_ncei = fetch_ncei_earthquakes(min_year=2000, max_year=2024)

print(f"\nShape: {df_ncei.shape}")
print(f"Columns: {df_ncei.columns.tolist()}")

  Page 8/8 — 1411 records so far
Done. Fetched 1411 total records (2000–2024)

Shape: (1411, 48)
Columns: ['id', 'year', 'month', 'day', 'hour', 'minute', 'second', 'locationName', 'latitude', 'longitude', 'eqDepth', 'eqMagnitude', 'damageAmountOrder', 'eqMagMb', 'publish', 'damageAmountOrderTotal', 'housesDamagedTotal', 'housesDamagedAmountOrderTotal', 'country', 'regionCode', 'injuries', 'injuriesAmountOrder', 'housesDestroyed', 'housesDestroyedAmountOrder', 'housesDamaged', 'housesDamagedAmountOrder', 'eqMagMw', 'eqMagMs', 'injuriesTotal', 'injuriesAmountOrderTotal', 'housesDestroyedTotal', 'housesDestroyedAmountOrderTotal', 'deaths', 'deathsAmountOrder', 'damageMillionsDollars', 'eqMagMl', 'deathsTotal', 'deathsAmountOrderTotal', 'damageMillionsDollarsTotal', 'tsunamiEventId', 'intensity', 'volcanoEventId', 'area', 'missing', 'missingAmountOrder', 'missingTotal', 'missingAmountOrderTotal', 'eqMagUnk']


In [32]:
print("Year range:", df_ncei["year"].min(), "–", df_ncei["year"].max())
print("Records per year:")
print(df_ncei["year"].value_counts().sort_index())

Year range: 2000 – 2024
Records per year:
year
2000    38
2001    26
2002    62
2003    73
2004    79
2005    60
2006    65
2007    67
2008    77
2009    62
2010    64
2011    62
2012    53
2013    56
2014    56
2015    48
2016    54
2017    66
2018    71
2019    64
2020    33
2021    42
2022    51
2023    46
2024    36
Name: count, dtype: int64


## 4. Build a proper datetime column

NCEI stores date as separate year/month/day columns. We combine them here
**before** any merge or filtering.

In [33]:
def build_datetime(row):
    try:
        year  = int(row["year"])
        month = int(row["month"]) if pd.notna(row.get("month")) else 1
        day   = int(row["day"])   if pd.notna(row.get("day"))   else 1
        return pd.Timestamp(year=year, month=month, day=day, tz="UTC")
    except Exception:
        return pd.NaT

df_ncei["time"] = df_ncei.apply(build_datetime, axis=1)

print("Null times:", df_ncei["time"].isna().sum())
df_ncei[["time", "latitude", "longitude", "eqMagnitude", "locationName"]].head()

Null times: 0


,time,latitude,longitude,eqMagnitude,locationName
0,2000-01-03 00:00:00+00:00,22.132,92.771,4.6,INDIA-BANGLADESH BORDER: MAHESHKHALI
1,2000-01-11 00:00:00+00:00,40.498,122.994,5.1,CHINA: LIAONING PROVINCE
2,2000-01-14 00:00:00+00:00,25.607,101.063,5.9,CHINA: YUNNAN PROVINCE: YAOAN COUNTY
3,2000-02-02 00:00:00+00:00,35.288,58.218,5.3,"IRAN: BARDASKAN, KASHMAR"
4,2000-02-07 00:00:00+00:00,-26.288,30.888,4.5,SOUTH AFRICA; SWAZILAND: MBABANE-MANZINI


## 5. Explore the impact columns

In [34]:
impact_cols = ["deaths", "injuries", "damageMillionsDollars", "housesDestroyed", "housesDamaged"]

for col in impact_cols:
    df_ncei[col] = pd.to_numeric(df_ncei[col], errors="coerce")

df_ncei[impact_cols].describe()

,deaths,injuries,damageMillionsDollars,housesDestroyed,housesDamaged
count,568.000000,759.000000,252.000000,3.890000e+02,4.400000e+02
mean,1133.098592,1639.156785,1808.007262,2.555153e+04,3.161228e+04
std,14387.959068,16578.609192,7011.318255,2.764507e+05,2.697799e+05
min,1.000000,1.000000,0.300000,1.000000e+00,1.000000e+00
25%,1.000000,6.000000,20.750000,2.100000e+01,1.000000e+02
50%,3.000000,26.000000,100.000000,2.170000e+02,6.605000e+02
75%,12.000000,132.500000,503.750000,1.756000e+03,4.125000e+03
max,316000.000000,374171.000000,86000.000000,5.360000e+06,5.360000e+06


In [35]:
# What fraction of records have each impact field populated?
df_ncei[impact_cols].notna().mean().sort_values(ascending=False).to_frame("pct_populated")

,pct_populated
injuries,0.537916
deaths,0.402551
housesDamaged,0.311836
housesDestroyed,0.275691
damageMillionsDollars,0.178597


In [36]:
# Deadliest earthquakes 2000–2024
df_ncei.nlargest(10, "deaths")[["year", "locationName", "eqMagnitude", "eqDepth", "deaths", "damageMillionsDollars"]]

,year,locationName,eqMagnitude,eqDepth,deaths,damageMillionsDollars
585,2010,HAITI: PORT-AU-PRINCE,7.0,13.0,316000.0,8000.0
465,2008,CHINA: SICHUAN PROVINCE,7.9,10.0,87652.0,86000.0
306,2005,"PAKISTAN: MUZAFFARABAD, URI, ANANTNAG, BARAMULA",7.6,15.0,76213.0,6680.0
1294,2023,TURKEY: KAHRAMANMARAS; SYRIA,7.8,17.0,56697.0,42900.0
182,2003,"IRAN: SOUTHEASTERN: BAM, BARAVAT",6.6,10.0,31000.0,500.0
36,2001,"INDIA: GUJARAT: BHUJ, AHMADABAD, RAJOKOT; PA...",7.6,17.0,20005.0,2623.0
867,2015,NEPAL: KATHMANDU; INDIA; CHINA; BANGLADESH,7.8,8.0,8957.0,6000.0
343,2006,"INDONESIA: JAVA: BANTUL, YOGYAKARTA",6.3,13.0,6234.0,3100.0
1071,2018,INDONESIA: SULAWESI,7.5,20.0,4340.0,1500.0
604,2010,CHINA: QINGHAI PROVINCE: YUSHU,6.9,17.0,2968.0,500.0


## 6. Save the full NCEI dataset

In [37]:
df_ncei.to_csv("ncei_earthquakes_2000_2024.csv", index=False)

In [34]:
from dateutil.relativedelta import relativedelta
from datetime import datetime, timezone
import time

def fetch_usgs_chunked(start_date: str, end_date: str, min_magnitude: float = 0) -> pd.DataFrame:
    """Fetch USGS events in 3-month chunks to avoid the 20k row limit."""
    USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    start = datetime.fromisoformat(start_date).replace(tzinfo=timezone.utc)
    end   = datetime.fromisoformat(end_date).replace(tzinfo=timezone.utc)

    frames = []
    current = start
    while current < end:
        chunk_end = min(current + relativedelta(months=1), end)
        params = {
            "format": "geojson",
            "starttime": current.date().isoformat(),
            "endtime":   chunk_end.date().isoformat(),
            "minmagnitude": min_magnitude,
            "limit": 20000,
            "orderby": "time-asc",
        }
        r = requests.get(USGS_URL, params=params)
        r.raise_for_status()
        features = r.json().get("features", [])
        rows = []
        for feat in features:
            props  = feat["properties"]
            coords = feat["geometry"]["coordinates"]
            rows.append({
                "usgs_id":   feat["id"],
                "time":      pd.Timestamp(props["time"], unit="ms", tz="UTC"),
                "latitude":  coords[1],
                "longitude": coords[0],
                "depth":     coords[2],
                "magnitude": props["mag"],
                "place":     props["place"],
                "sig":       props["sig"],
                "mmi":       props["mmi"],
                "alert":     props["alert"],
            })
        frames.append(pd.DataFrame(rows))
        print(f"  {current.date()} → {chunk_end.date()}: {len(rows)} records", end="\r")
        current = chunk_end
        time.sleep(0.5)

    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["usgs_id"])
    print(f"\nDone. {len(df)} total USGS records fetched.")
    return df


df_usgs = fetch_usgs_chunked("2000-01-01", "2024-12-31", min_magnitude=0)
df_usgs.head(3)

  2024-12-01 → 2024-12-31: 11657 records
Done. 3035653 total USGS records fetched.


,usgs_id,time,latitude,longitude,depth,magnitude,place,sig,mmi,alert
0,ci9131988,2000-01-01 00:02:46.200000+00:00,34.692000,-116.355000,11.271,0.80,"18km W of Ludlow, California",10,NaN,NaN
1,ci9521550,2000-01-01 00:03:08+00:00,34.439333,-116.221667,5.981,0.70,"32km S of Ludlow, California",8,NaN,NaN
2,nc21075021,2000-01-01 00:03:53.650000+00:00,37.416667,-121.766500,5.360,1.23,"5 km NE of East Foothills, California",23,NaN,NaN


In [3]:
# Save USGS too
df_usgs.to_csv("usgs_earthquakes_2000_2024.csv", index=False)
print(f"Saved {len(df_usgs)} records to usgs_earthquakes_2000_2024.csv")

NameError: name 'df_usgs' is not defined

In [38]:
df_usgs = pd.read_csv("usgs_earthquakes_2000_2024.csv", parse_dates=["time"])
print(f"Reloaded {len(df_usgs)} USGS records from CSV")
df_usgs.head(3)

Reloaded 3035653 USGS records from CSV


,usgs_id,time,latitude,longitude,depth,magnitude,place,sig,mmi,alert
0,ci9131988,2000-01-01 00:02:46.200000+00:00,34.692000,-116.355000,11.271,0.80,"18km W of Ludlow, California",10,NaN,NaN
1,ci9521550,2000-01-01 00:03:08+00:00,34.439333,-116.221667,5.981,0.70,"32km S of Ludlow, California",8,NaN,NaN
2,nc21075021,2000-01-01 00:03:53.650000+00:00,37.416667,-121.766500,5.360,1.23,"5 km NE of East Foothills, California",23,NaN,NaN


In [39]:
df_usgs["time"] = pd.to_datetime(df_usgs["time"], errors="coerce", utc=True)

## 7. Prototype the USGS ↔ NCEI merge

Strategy: for each NCEI significant earthquake, find the closest USGS event by:
1. Time window: ±3 days
2. Spatial proximity: within ~1 degree lat/lon (≈111 km)

In [40]:
def merge_usgs_ncei(df_usgs, df_ncei, time_tolerance_days=3, coord_tolerance_deg=1.0):
    """
    Left-join NCEI significant earthquakes onto USGS events.
    For each NCEI record, find the best-matching USGS record by:
      - time within ±time_tolerance_days
      - lat/lon within ±coord_tolerance_deg
      - closest eqMagnitude as tiebreaker
    """
    time_delta = pd.Timedelta(days=time_tolerance_days)
    matched = []

    for _, ncei_row in df_ncei.iterrows():
        # Skip rows with no time or location
        if pd.isna(ncei_row["time"]) or pd.isna(ncei_row["latitude"]):
            continue

        # Filter USGS to time window
        time_mask = (
            (df_usgs["time"] >= ncei_row["time"] - time_delta) &
            (df_usgs["time"] <= ncei_row["time"] + time_delta)
        )
        candidates = df_usgs[time_mask].copy()
        if candidates.empty:
            continue

        # Filter by spatial proximity
        candidates = candidates[
            (np.abs(candidates["latitude"]  - ncei_row["latitude"])  <= coord_tolerance_deg) &
            (np.abs(candidates["longitude"] - ncei_row["longitude"]) <= coord_tolerance_deg)
        ]
        if candidates.empty:
            continue

        # Pick closest by magnitude — NOTE: NCEI column is eqMagnitude, not magnitude
        if pd.notna(ncei_row.get("eqMagnitude")):
            best_idx = (candidates["magnitude"] - ncei_row["eqMagnitude"]).abs().idxmin()
        else:
            best_idx = candidates.index[0]

        usgs_match = candidates.loc[best_idx]
        row = {
            **{f"ncei_{k}": v for k, v in ncei_row.items()},
            **{f"usgs_{k}": v for k, v in usgs_match.items()},
        }
        matched.append(row)

    result = pd.DataFrame(matched)
    print(f"Matched {len(result)} / {len(df_ncei)} NCEI records to a USGS event ({len(result)/len(df_ncei):.1%})")
    return result


merged = merge_usgs_ncei(df_usgs, df_ncei)
merged.head(3)

Matched 1378 / 1411 NCEI records to a USGS event (97.7%)


,ncei_id,ncei_year,ncei_month,ncei_day,ncei_hour,ncei_minute,ncei_second,ncei_locationName,ncei_latitude,ncei_longitude,...,usgs_usgs_id,usgs_time,usgs_latitude,usgs_longitude,usgs_depth,usgs_magnitude,usgs_place,usgs_sig,usgs_mmi,usgs_alert
0,5551,2000,1,3,22.0,34.0,12.6,INDIA-BANGLADESH BORDER: MAHESHKHALI,22.132,92.771,...,usp0009kqm,2000-01-03 22:34:12.640000+00:00,22.132,92.771,33.0,4.6,"45 km SSW of Saiha, India",326,3.700,NaN
1,5552,2000,1,11,23.0,43.0,56.4,CHINA: LIAONING PROVINCE,40.498,122.994,...,usp0009m1j,2000-01-11 23:43:56.450000+00:00,40.498,122.994,10.0,5.1,"Liaoning, China",400,4.600,NaN
2,5553,2000,1,14,23.0,37.0,7.8,CHINA: YUNNAN PROVINCE: YAOAN COUNTY,25.607,101.063,...,usp0009m4u,2000-01-14 23:37:07.870000+00:00,25.607,101.063,33.0,5.9,"85 km E of Dali, China",536,5.928,NaN


In [41]:
# Sanity check: do the two sources agree on magnitude?
merged["magnitude_diff"] = (merged["usgs_magnitude"] - merged["ncei_eqMagnitude"]).abs()
print(merged["magnitude_diff"].describe())
print(f"\nMatches within 0.3 magnitude units: {(merged['magnitude_diff'] < 0.3).mean():.1%}")
print(f"Matches within 0.5 magnitude units: {(merged['magnitude_diff'] < 0.5).mean():.1%}")

count    1375.000000
mean        0.050065
std         0.216088
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         2.700000
Name: magnitude_diff, dtype: float64

Matches within 0.3 magnitude units: 96.3%
Matches within 0.5 magnitude units: 97.8%


In [42]:
# Save the merged dataset
merged.to_csv("merged_earthquakes_2000_2024.csv", index=False)
print(f"Saved {len(merged)} merged records.")

Saved 1378 merged records.


In [43]:
merged[["ncei_year", "ncei_locationName", "ncei_eqMagnitude", "usgs_place", "usgs_magnitude", "magnitude_diff"]].head(10)

,ncei_year,ncei_locationName,ncei_eqMagnitude,usgs_place,usgs_magnitude,magnitude_diff
0,2000,INDIA-BANGLADESH BORDER: MAHESHKHALI,4.6,"45 km SSW of Saiha, India",4.6,0.0
1,2000,CHINA: LIAONING PROVINCE,5.1,"Liaoning, China",5.1,0.0
2,2000,CHINA: YUNNAN PROVINCE: YAOAN COUNTY,5.9,"85 km E of Dali, China",5.9,0.0
3,2000,"IRAN: BARDASKAN, KASHMAR",5.3,"22 km E of Bardaskan, Iran",5.3,0.0
4,2000,SOUTH AFRICA; SWAZILAND: MBABANE-MANZINI,4.5,"22 km NW of Mhlambanyatsi, Eswatini",4.5,0.0
5,2000,JAPAN: VOLCANO ISLANDS,7.6,"Volcano Islands, Japan region",7.6,0.0
6,2000,"INDONESIA: SULAWESI: LUWUK, BANGGAI, PELENG,",7.6,"89 km E of Luwuk, Indonesia",7.6,0.0
7,2000,"TURKEY: DOGANYOL, PUTURGE",4.1,"20 km NE of Sincik, Turkey",4.2,0.1
8,2000,TAIWAN: TAI-CHUNG COUNTY,5.4,"29 km NNE of Puli, Taiwan",5.4,0.0
9,2000,"INDONESIA: SUMATRA: BENGKULU, ENGGANO",7.9,"103 km S of Bengkulu, Indonesia",7.9,0.0


In [44]:
merged[["ncei_damageAmountOrderTotal", "ncei_housesDamagedTotal", "ncei_housesDamagedAmountOrderTotal", "ncei_locationName", "ncei_eqMagnitude", "usgs_place", "usgs_magnitude", "magnitude_diff"]].head(10)

,ncei_damageAmountOrderTotal,ncei_housesDamagedTotal,ncei_housesDamagedAmountOrderTotal,ncei_locationName,ncei_eqMagnitude,usgs_place,usgs_magnitude,magnitude_diff
0,1.0,100.0,2.0,INDIA-BANGLADESH BORDER: MAHESHKHALI,4.6,"45 km SSW of Saiha, India",4.6,0.0
1,3.0,8800.0,4.0,CHINA: LIAONING PROVINCE,5.1,"Liaoning, China",5.1,0.0
2,4.0,NaN,NaN,CHINA: YUNNAN PROVINCE: YAOAN COUNTY,5.9,"85 km E of Dali, China",5.9,0.0
3,2.0,300.0,3.0,"IRAN: BARDASKAN, KASHMAR",5.3,"22 km E of Bardaskan, Iran",5.3,0.0
4,1.0,NaN,NaN,SOUTH AFRICA; SWAZILAND: MBABANE-MANZINI,4.5,"22 km NW of Mhlambanyatsi, Eswatini",4.5,0.0
5,NaN,NaN,NaN,JAPAN: VOLCANO ISLANDS,7.6,"Volcano Islands, Japan region",7.6,0.0
6,4.0,NaN,NaN,"INDONESIA: SULAWESI: LUWUK, BANGGAI, PELENG,",7.6,"89 km E of Luwuk, Indonesia",7.6,0.0
7,2.0,200.0,3.0,"TURKEY: DOGANYOL, PUTURGE",4.1,"20 km NE of Sincik, Turkey",4.2,0.1
8,1.0,NaN,NaN,TAIWAN: TAI-CHUNG COUNTY,5.4,"29 km NNE of Puli, Taiwan",5.4,0.0
9,3.0,NaN,NaN,"INDONESIA: SUMATRA: BENGKULU, ENGGANO",7.9,"103 km S of Bengkulu, Indonesia",7.9,0.0


In [45]:
cols = [
    "ncei_damageAmountOrderTotal",
    "ncei_housesDamagedTotal",
    "ncei_housesDamagedAmountOrderTotal",
]

missing = merged[cols].isna().sum().rename("missing_count")
print(missing)

ncei_damageAmountOrderTotal           262
ncei_housesDamagedTotal               952
ncei_housesDamagedAmountOrderTotal    711
Name: missing_count, dtype: int64


In [46]:
# How many records each source started with
print("=== Before merge ===")
print(f"NCEI records:  {len(df_ncei)}")
print(f"USGS records:  {len(df_usgs)}")

# How many NCEI records made it into the merge
ncei_matched = len(merged)
ncei_unmatched = len(df_ncei) - ncei_matched
print(f"\n=== After merge ===")
print(f"Matched rows:       {ncei_matched}  ({ncei_matched/len(df_ncei)*100:.1f}% of NCEI)")
print(f"Unmatched NCEI:     {ncei_unmatched}  ({ncei_unmatched/len(df_ncei)*100:.1f}% of NCEI)")

# Why did NCEI records fail to match?
# Rebuild the unmatched set by finding NCEI ids not in the merge
matched_ncei_ids = merged["ncei_id"].values
unmatched = df_ncei[~df_ncei["id"].isin(matched_ncei_ids)].copy()

print(f"\n=== Unmatched NCEI records breakdown ===")
print(f"Missing lat/lon:   {unmatched[['latitude','longitude']].isna().any(axis=1).sum()}")
print(f"Missing time:      {unmatched['time'].isna().sum()}")
print(f"No USGS match:     {len(unmatched)}")

# Where are the unmatched ones from?
print(f"\nTop countries in unmatched records:")
print(unmatched["country"].value_counts().head(10))

# What magnitudes are unmatched?

print(f"\nMagnitude distribution of unmatched records:")
unmatched["eqMagnitude"] = pd.to_numeric(unmatched["eqMagnitude"], errors="coerce")

print(pd.cut(unmatched["eqMagnitude"], bins=[0, 3, 4, 5, 6]).value_counts().sort_index())

=== Before merge ===
NCEI records:  1411
USGS records:  3035653

=== After merge ===
Matched rows:       1378  (97.7% of NCEI)
Unmatched NCEI:     33  (2.3% of NCEI)

=== Unmatched NCEI records breakdown ===
Missing lat/lon:   0
Missing time:      0
No USGS match:     33

Top countries in unmatched records:
country
IRAN           4
TURKEY         3
AUSTRALIA      3
INDIA          2
PHILIPPINES    2
TAJIKISTAN     1
NEW ZEALAND    1
MALAWI         1
BRAZIL         1
ARGENTINA      1
Name: count, dtype: int64

Magnitude distribution of unmatched records:
eqMagnitude
(0, 3]     0
(3, 4]    11
(4, 5]    10
(5, 6]     9
Name: count, dtype: int64


In [47]:
print(unmatched["eqMagnitude"].describe())
print(unmatched["eqMagnitude"].unique()[:20])


count    33.000000
mean      4.639394
std       1.040114
min       3.100000
25%       3.800000
50%       4.500000
75%       5.200000
max       7.200000
Name: eqMagnitude, dtype: float64
[5.9 4.8 4.1 4.4 3.8 5.1 4.3 5.  3.1 5.2 4.9 3.7 4.  7.2 4.2 4.5 3.6 6.1
 3.4 3.2]


In [48]:
matched_ncei_ids = merged["ncei_id"].values
unmatched = df_ncei[~df_ncei["id"].isin(matched_ncei_ids)].copy()
unmatched["eqMagnitude"] = pd.to_numeric(unmatched["eqMagnitude"], errors="coerce")

print(f"Unmatched: {len(unmatched)}")

print("\nMagnitude distribution:")
print(pd.cut(unmatched["eqMagnitude"], bins=[0,3,4,5,6,7,8,10]).value_counts().sort_index())

print("\nTop countries:")
print(unmatched["country"].value_counts().head(10))

print("\nYear distribution:")
print(unmatched["year"].value_counts().sort_index())

print("\nMissing lat/lon:")
print(unmatched[["latitude", "longitude"]].isna().sum())

Unmatched: 33

Magnitude distribution:
eqMagnitude
(0, 3]      0
(3, 4]     11
(4, 5]     10
(5, 6]      9
(6, 7]      2
(7, 8]      1
(8, 10]     0
Name: count, dtype: int64

Top countries:
country
IRAN           4
TURKEY         3
AUSTRALIA      3
INDIA          2
PHILIPPINES    2
TAJIKISTAN     1
NEW ZEALAND    1
MALAWI         1
BRAZIL         1
ARGENTINA      1
Name: count, dtype: int64

Year distribution:
year
2001    1
2002    1
2003    1
2005    1
2006    2
2007    1
2008    3
2009    2
2010    4
2011    1
2012    1
2013    1
2014    2
2015    2
2017    1
2018    3
2019    6
Name: count, dtype: int64

Missing lat/lon:
latitude     0
longitude    0
dtype: int64
